## Agenda

1. Macro variables: when to use quotes
2. Formatting values with `FORMAT`
3. Sorting and removing duplicates with `PROC SORT`
4. Preparing data with the DATA step: filter, subset, compute
5. Functions

## Macro variables: no quotes in `%LET`

From last class's exercise: a macro variable is **always text**, so you don't need quotes in `%LET`.
If you add them, the quotes become **part of the value**: they are taken *literally*.

```{=html}
<div class="macro-compare">
<div class="anno"><div class="anno-row"><pre class="anno-code"><span class="cmt">/* quotes are part of the value */</span>
<span class="kw">%let</span> p = <span class="libref-circle">"player1"</span>;

<span class="kw">where</span> Player = "&amp;p";
<span class="cmt">/* becomes: Player = ""player1"" */</span></pre></div></div>
<div class="macro-arrow fragment" data-fragment-index="1"><div class="arrow-right black"></div></div>
<div class="anno fragment" data-fragment-index="1"><div class="anno-row"><pre class="anno-code"><span class="cmt">/* no quotes: just the text */</span>
<span class="kw">%let</span> p = player1;

<span class="kw">where</span> Player = "&amp;p";
<span class="cmt">/* becomes: Player = "player1" */</span></pre></div></div>
</div>
```

::: {.fragment .red-text}
Leave the quotes off in `%LET`, and add them where you **use** the macro variable.
:::

## Using a macro variable

```{=html}
<div class="syntax-box"><b>%LET</b> <i>macrovar</i>=<i>value</i>;<br><br>
&nbsp;&nbsp;&nbsp;&nbsp;<i>numvar</i>=&amp;<i>macrovar</i>; &nbsp;&nbsp;<span class="cmt">/* number */</span><br>
&nbsp;&nbsp;&nbsp;&nbsp;<i>charvar</i>="&amp;<i>macrovar</i>"; &nbsp;&nbsp;<span class="cmt">/* character */</span><br>
&nbsp;&nbsp;&nbsp;&nbsp;<i>datevar</i>="&amp;<i>macrovar</i>"d; &nbsp;&nbsp;<span class="cmt">/* date */</span></div>
```

::: {.incremental}
- A macro variable stores a **text string** that is substituted into your program.
- `%LET` names the macro variable and assigns its value. Names follow the usual SAS naming rules.
- Reference it with an `&` (ampersand) in front of the name.
- Inside quotation marks, use **double** quotes: `"&macrovar"`.
:::

In [ ]:
%let start_date = 01JAN2005;

proc print data=sashelp.stocks (obs=5);
    where Date >= "&start_date"d;
run;

Obs,Stock,Date,Open,High,Low,Close,Volume,AdjClose
1,IBM,01DEC05,$89.15,$89.92,$81.56,$82.20,"5,976,252",$81.37
2,IBM,01NOV05,$81.85,$89.94,$80.64,$88.90,"5,556,471",$88.01
3,IBM,03OCT05,$80.22,$84.60,$78.70,$81.88,"7,019,666",$80.86
4,IBM,01SEP05,$80.16,$82.11,$76.93,$80.22,"5,772,280",$79.22
5,IBM,01AUG05,$83.00,$84.20,$79.87,$80.62,"4,801,386",$79.62


## `FORMAT`

A **format** changes how a value is **displayed**, not the value that is stored: `12345.678` → `12,345.7`.

```{=html}
<div class="anno"><div class="anno-row"><pre class="anno-code"><span class="kw">proc print</span> data=<i>...</i>;
    <span class="kw">format</span> <i>col-name(s)</i> <span class="libref-circle"><i>format</i></span>;<span class="anno-inline fragment"><span class="arrow-left"></span><span class="anno-inline-text"><span class="mono">&lt;$&gt;format-name&lt;w&gt;.&lt;d&gt;</span><br><b>$</b>: character format<br><b>w</b>: total width of the formatted value<br><b>d</b>: number of decimal places (numeric formats)</span></span>
<span class="kw">run</span>;</pre></div></div>
```

::: {.fragment}
:::: {.columns}
::: {.column width="48%"}
**Numeric values**

| Format | Value | Applied | Displays |
|---|---|---|---|
| *w.d* | 20435.86 | `5.` | 20436 |
| `COMMA`*w.d* | 20435.86 | `COMMA8.1` | 20,435.9 |
| `DOLLAR`*w.d* | 20435.86 | `DOLLAR10.` | $20,436 |
:::
::: {.column width="52%"}
**Date values** (days since Jan 1, 1960)

| Value | Applied | Displays |
|---|---|---|
| 21199 | `DATE7.` | 15JAN18 |
| 21199 | `MMDDYY10.` | 01/15/2018 |
| 21199 | `WEEKDATE.` | Monday, January 15, 2018 |
:::
::::

More formats: [SAS formats by category](https://documentation.sas.com/doc/en/pgmsascdc/9.4_3.5/allprodsle/syntaxByType-format.htm)
:::

## `FORMAT`: numbers

Add commas to `Damage` and `Heals` in the Overwatch stats from last class.

In [ ]:
libname ma505 "~";    /* skip if your autoexec does this */

proc print data=ma505.ow_stats (obs=10);
    format Damage comma10. Heals comma10.;
run;

Obs,Player,Date,Win,Map,Mode,Role,Elimination,Assists,Deaths,Damage,Heals,Mitigation
1,player1,08/15/2024,1,Suravasa,Assault,Damage,28,0,5,"11,480","2,484",0
2,player1,08/15/2024,1,King's Row,Attack Escort,Damage,17,3,4,"6,713",0,245
3,player1,08/15/2024,0,New Queen Street,Robot,Tank,27,4,7,"5,776",122,2082
4,player1,08/15/2024,0,Midtown,Attack Escort,Damage,8,3,6,"7,138",8,828
5,player1,08/15/2024,1,Ilios,Attack,Support,10,3,4,"2,163","4,849",0
6,player1,08/15/2024,1,Havana,Escort,Tank,7,2,9,"4,416","2,498",1068
7,player1,08/14/2024,0,Blizzard World,Attack Escort,Tank,22,6,9,"7,538",0,5654
8,player1,08/14/2024,1,Junkertown,Escort,Tank,15,3,7,"10,948","2,260",6078
9,player1,08/14/2024,1,New Junk City,Assault,Tank,27,8,4,"9,735",225,2012
10,player1,08/14/2024,1,Paraiso,Attack Escort,Damage,8,5,1,"4,102",0,0


## `FORMAT`: dates

`Date` is stored as a number. The format only changes how it looks.

In [ ]:
proc print data=ma505.ow_stats (obs=5);
    var Player Date;
    format Date weekdate.;
run;

Obs,Player,Date
1,player1,"Thursday, August 15, 2024"
2,player1,"Thursday, August 15, 2024"
3,player1,"Thursday, August 15, 2024"
4,player1,"Thursday, August 15, 2024"
5,player1,"Thursday, August 15, 2024"


## Sorting data: `PROC SORT`

```{=html}
<div class="anno"><div class="anno-row"><pre class="anno-code"><span class="kw">proc sort</span> data=<i>input-table</i> <span class="libref-circle">out=<i>output-table</i></span>;<span class="anno-inline fragment"><span class="arrow-left"></span><span class="anno-inline-text">If you leave out <span class="mono">OUT=</span>, SAS <b>overwrites</b> the input table with the sorted version.</span></span>
    <span class="kw">by</span> <span class="libref-circle">&lt;descending&gt;</span><span class="anno-inline fragment"><span class="arrow-left"></span><span class="anno-inline-text">Sorting is <b>ascending</b> by default. <span class="mono">DESCENDING</span> flips it for the column <b>right after it</b>.</span></span>
        <i>col-name(s)</i>;
<span class="kw">run</span>;</pre></div></div>
```

## `PROC SORT`: example

`out=test_sort` creates a new table so we do not overwrite the original one. When sorting, `descending` only applies to `Age`. If there is a tie in `Age`, then it sorts by `Name`.

In [ ]:
proc sort data=sashelp.class out=test_sort;
    by descending Age Name;
run;

proc print data=test_sort;
run;

Obs,Name,Sex,Age,Height,Weight
1,Philip,M,16,72.0,150.0
2,Janet,F,15,62.5,112.5
3,Mary,F,15,66.5,112.0
4,Ronald,M,15,67.0,133.0
5,William,M,15,66.5,112.0
6,Alfred,M,14,69.0,112.5
7,Carol,F,14,62.8,102.5
8,Henry,M,14,63.5,102.5
9,Judy,F,14,64.3,90.0
10,Alice,F,13,56.5,84.0


## Removing duplicate rows

```{=html}
<div class="anno"><div class="anno-row"><pre class="anno-code"><span class="kw">proc sort</span> data=<i>input-table</i> out=<i>output-table</i>
    <span class="libref-circle"><span class="kw">nodupkey</span></span><span class="anno-inline fragment"><span class="arrow-left"></span><span class="anno-inline-text">Removes rows with duplicate <span class="mono">BY</span> values that sit next to each other, keeping only the <b>first</b> one.</span></span>
    <span class="libref-circle">&lt;dupout=<i>dup-table</i>&gt;</span>;<span class="anno-inline fragment"><span class="arrow-left"></span><span class="anno-inline-text">Optional: a table that collects the duplicates that were removed.</span></span>
    <span class="kw">by</span> <span class="libref-circle">_all_</span>;<span class="anno-inline fragment"><span class="arrow-left"></span><span class="anno-inline-text">Sorts by <b>all</b> columns, so identical rows end up next to each other. You can also list one or more columns.</span></span>
<span class="kw">run</span>;</pre></div></div>
```

## Removing duplicates: example

First, make a copy of `sashelp.class` with two rows (Jane and Judy) repeated. Just run this; we'll learn the DATA step later today.

In [ ]:
libname ma505 "~";    /* skip if your autoexec does this */

/* setup: sashelp.class plus 2 duplicate rows (21 rows) */
data ma505.class;
    set sashelp.class sashelp.class(where=(Name in ("Jane", "Judy")));
run;

Now remove the duplicates. `ma505.clean` gets 19 unique rows and `ma505.duplicates` gets the 2 removed rows.

In [ ]:
proc sort data=ma505.class out=ma505.clean
    nodupkey
    dupout=ma505.duplicates;
    by _all_;
run;

proc print data=ma505.duplicates;
run;

Obs,Name,Sex,Age,Height,Weight
1,Jane,F,12,59.8,84.5
2,Judy,F,14,64.3,90.0


## Exercise

Use the Overwatch heroes table `ma505.overwatch` from Day 2 (re-run the Day 2 import if it's missing).

1. Sort it by `Player`, `Date`, `Win`, `Map`, `Role`, and `Percent Played` (descending). Save the result as `ma505.ow_hrsrt`.
    a. `Percent Played` has a space, so type it as a **name literal**: `"Percent Played"n`
2. Using the sorted data, remove duplicate rows by `Player`, `Date`, `Win`, `Map`, and `Role` only.
3. What **hero** is in observation 14 of the de-duplicated data?
    a. Tip: `proc print data=... (firstobs=14 obs=14);` prints just row 14.

<div class="solution-start"></div>

In [ ]:
libname ma505 "~";    /* skip if your autoexec does this */

/* 1. sort, highest Percent Played first within each game */
proc sort data=ma505.overwatch out=ma505.ow_hrsrt;
    by Player Date Win Map Role descending "Percent Played"n;
run;

/* 2. keep the first row of each Player/Date/Win/Map/Role */
proc sort data=ma505.ow_hrsrt out=ma505.hero_clean
    nodupkey
    dupout=ma505.hr_dup;
    by Player Date Win Map Role;
run;

/* 3. print only row 14 */
proc print data=ma505.hero_clean (firstobs=14 obs=14);
run;

Obs,Player,Date,Win,Map,Mode,Role,Percent Played,Hero
14,player1,08/13/2024,0,Watchpoint: Gibr,Escort,Damage,56,Hanzo


Observation 14 of the de-duplicated data is **Hanzo** (player1, 08/13/2024, Watchpoint: Gibraltar, Damage, 56% played).


## Preparing data with the DATA step

What we want to do with our data:

::: {.incremental}
- **manipulate**
- **filter**
- **process**
- **compute**
:::

## The DATA step creates a SAS table

```{=html}
<div class="anno"><div class="anno-row"><pre class="anno-code"><span class="kw">data</span> <span class="libref-circle"><i>output-table</i></span>;<span class="anno-inline fragment"><span class="arrow-left"></span><span class="anno-inline-text">the table to <b>create</b></span></span>
    <span class="kw">set</span> <span class="libref-circle"><i>input-table</i></span>;<span class="anno-inline fragment"><span class="arrow-left"></span><span class="anno-inline-text">the table to <b>read</b></span></span>
<span class="kw">run</span>;</pre></div></div>
```

In [ ]:
/* creating a NEW table using a DATA step */
data cln_class;
    set ma505.clean;
run;

This reads the de-duplicated `ma505.clean` table and writes a copy called `cln_class`. There's no libref, so it goes in the **work** library. Check the log: 19 observations read, and `WORK.CLN_CLASS` has 19 observations.

## How does the DATA step process?

```{=html}
<div class="ds-phases">
  <div class="ds-box comp"><h4>Compilation</h4><ul><li>Checks for syntax errors</li><li>Identifies the column attributes</li><li>Establishes the table metadata</li></ul></div>
  <div class="ds-arrow"><div class="arrow-right black"></div></div>
  <div class="ds-box exec"><h4>Execution</h4><ul><li>Reads and writes data</li><li>Performs calculations and manipulations</li></ul></div>
</div>
```

::: {.fragment}
During execution, the DATA step **loops once per row**:

1. Reads a row from the input table (`ma505.clean`)
2. Processes each statement, top to bottom
3. Writes the row to the output table
4. Loops back to the top to read the next row
:::

## Filtering rows in the DATA step

```{=html}
<div class="syntax-box"><b>DATA</b> <i>output-table</i>;<br>
&nbsp;&nbsp;&nbsp;&nbsp;<b>SET</b> <i>input-table</i>;<br>
&nbsp;&nbsp;&nbsp;&nbsp;<u><b>WHERE</b> <i>expression</i>;</u><br>
<b>RUN;</b></div>
```

Same `WHERE` as in the procedures, but now the result is saved as a **new table**.

In [ ]:
/* keep only students older than 14 */
data cln_class;
    set ma505.clean;
    where Age > 14;
run;

proc print data=cln_class;
run;

Obs,Name,Sex,Age,Height,Weight
1,Janet,F,15,62.5,112.5
2,Mary,F,15,66.5,112.0
3,Philip,M,16,72.0,150.0
4,Ronald,M,15,67.0,133.0
5,William,M,15,66.5,112.0


## Keeping or dropping columns

```{=html}
<div class="syntax-box"><b>DATA</b> <i>output-table</i>;<br>
&nbsp;&nbsp;&nbsp;&nbsp;<b>SET</b> <i>input-table</i>;<br>
&nbsp;&nbsp;&nbsp;&nbsp;<u><b>DROP</b> <i>col-name(s)</i>;</u> &nbsp;<span class="cmt">/* columns to remove */</span><br>
&nbsp;&nbsp;&nbsp;&nbsp;<u><b>KEEP</b> <i>col-name(s)</i>;</u> &nbsp;<span class="cmt">/* columns to keep */</span><br>
<b>RUN;</b></div>
```

Use one or the other: `KEEP` when you want a few columns, `DROP` when you want to lose a few.

In [ ]:
/* older than 14, and only three columns */
data cln_class;
    set ma505.clean;
    where Age > 14;
    keep Name Age Weight;
run;

proc print data=cln_class;
run;

Obs,Name,Age,Weight
1,Janet,15,112.5
2,Mary,15,112.0
3,Philip,16,150.0
4,Ronald,15,133.0
5,William,15,112.0


## Computing new columns

```{=html}
<div class="syntax-box"><b>DATA</b> <i>output-table</i>;<br>
&nbsp;&nbsp;&nbsp;&nbsp;<b>SET</b> <i>input-table</i>;<br>
&nbsp;&nbsp;&nbsp;&nbsp;<u><i>newcol</i> = <i>expression</i>;</u><br>
<b>RUN;</b></div>
```

A new column can be a calculation (`Profit`) or a constant (`Source`). `FORMAT` works inside the DATA step too, and it's saved with the table.

In [ ]:
/* computing new columns in the DATA step */
data cars_new;
    set sashelp.cars;
    where Origin ne "USA";
    Profit = MSRP - Invoice;
    Source = "Non-US Cars";
    format Profit dollar10.;
    keep Make Model Profit Source;
run;

proc print data=cars_new (obs=10);
run;

Obs,Make,Model,Profit,Source
1,Acura,MDX,"$3,608",Non-US Cars
2,Acura,RSX Type S 2dr,"$2,059",Non-US Cars
3,Acura,TSX 4dr,"$2,343",Non-US Cars
4,Acura,TL 4dr,"$2,896",Non-US Cars
5,Acura,3.5 RL 4dr,"$4,741",Non-US Cars
6,Acura,3.5 RL w/Navigation 4dr,"$5,000",Non-US Cars
7,Acura,NSX coupe 2dr manual S,"$9,787",Non-US Cars
8,Audi,A4 1.8T 4dr,"$2,432",Non-US Cars
9,Audi,A41.8T convertible 2dr,"$3,434",Non-US Cars
10,Audi,A4 3.0 4dr,"$2,994",Non-US Cars


## Functions

```{=html}
<div class="syntax-box"><b>DATA</b> <i>output-table</i>;<br>
&nbsp;&nbsp;&nbsp;&nbsp;<b>SET</b> <i>input-table</i>;<br>
&nbsp;&nbsp;&nbsp;&nbsp;<u><i>newcol</i> = <i>function</i>(<i>arguments</i>);</u><br>
<b>RUN;</b></div>
```

::: {.fragment}
:::: {.columns}
::: {.column width="22%"}
| Numeric |
|---|
| `SUM` |
| `MEAN` |
| `MEDIAN` |
| `RANGE` |
| `MIN` / `MAX` |
| `NMISS` |
:::
::: {.column width="38%"}
| Character | |
|---|---|
| `UPCASE` | all uppercase |
| `LOWCASE` | all lowercase |
| `PROPCASE` | First Letter Of Each Word |
| `CATS` | joins strings together |
| `SUBSTR` | part of a string |
:::
::: {.column width="40%"}
| Date | |
|---|---|
| `MONTH` | month, 1–12 |
| `YEAR` | four-digit year |
| `DAY` | day of the month, 1–31 |
| `WEEKDAY` | day of the week, 1–7 |
| `QTR` | quarter, 1–4 |
| `TODAY()` | today's date |
| `MDY` | date from month, day, year |
:::
::::
:::

## Functions: example

In [ ]:
/* using functions */
data cars_new;
    set sashelp.cars;
    MPG = sum(MPG_City, MPG_Highway);
    MPGavg = mean(MPG_City, MPG_Highway);
    Make = upcase(Make);
    Type = upcase(Type);
run;

proc print data=cars_new (obs=10);
    var Make Model Type MPG_City MPG_Highway MPG MPGavg;
run;

Obs,Make,Model,Type,MPG_City,MPG_Highway,MPG,MPGavg
1,ACURA,MDX,SUV,17,23,40,20.0
2,ACURA,RSX Type S 2dr,SEDAN,24,31,55,27.5
3,ACURA,TSX 4dr,SEDAN,22,29,51,25.5
4,ACURA,TL 4dr,SEDAN,20,28,48,24.0
5,ACURA,3.5 RL 4dr,SEDAN,18,24,42,21.0
6,ACURA,3.5 RL w/Navigation 4dr,SEDAN,18,24,42,21.0
7,ACURA,NSX coupe 2dr manual S,SPORTS,17,24,41,20.5
8,AUDI,A4 1.8T 4dr,SEDAN,22,31,53,26.5
9,AUDI,A41.8T convertible 2dr,SEDAN,23,30,53,26.5
10,AUDI,A4 3.0 4dr,SEDAN,20,28,48,24.0
